# BERT Text Classification — PyTorch (Patched & Flexible)

This notebook fixes variable inconsistencies, column name mismatches, and data-loading logic. It supports:

- **Single CSV** workflow (train/val/test split inside the notebook), or
- **Three CSVs** workflow (explicit `TRAIN_PATH`, `VAL_PATH`, `TEST_PATH`).
- Configurable column names: `TEXT_COL`, `LABEL_COL`, `SOURCE_COL`.
- Optional **real-only validation/test** via `REAL_ONLY_EVAL`.
- Label encoding for string labels and saved `label_map.json`.
- Class-weighted loss, early stopping on macro-F1, warmup + linear decay, gradient clipping.


In [ ]:

%pip install -U transformers datasets scikit-learn evaluate torch --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
from collections import Counter
from dataclasses import dataclass

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

/Users/prathamsaurabh/Authenticator.ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cpu'

## Config
Choose **one** data-loading mode below and set column names.

In [ ]:
# ---- Data Loading Mode ----
SINGLE_CSV = False          # Use your pre-split CSV files
REAL_ONLY_EVAL = True       # If True, validation/test sets are drawn from rows where source_type == 'real'

# ---- Paths ----
DATA_PATH = 'data.csv'      # Used only if SINGLE_CSV=True
TRAIN_PATH = 'hierarchical_train.csv'    # Hierarchical training data
VAL_PATH   = 'hierarchical_val.csv'      # Hierarchical validation data
TEST_PATH  = 'hierarchical_test.csv'     # Hierarchical test data

# ---- Column Names ----
TEXT_COL   = 'cleaned_text' # Your text column name
LABEL_COL  = 'hierarchical_label'     # Hierarchical label column name
SOURCE_COL = 'source_type'  # 'real' | 'synthetic' | 'mixed' (optional; can be missing)

# ---- Model & Training ----
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 256
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 4
WARMUP_RATIO = 0.1
PATIENCE = 3
WEIGHT_DECAY = 0.01


## Load Data
This block handles both **single CSV** and **three CSV** workflows. It also aligns column names and optionally enforces real-only evaluation.

In [ ]:
def standardize_columns(df, text_col, label_col, source_col):
    # Ensure required columns exist
    assert text_col in df.columns, f'Missing text column: {text_col}'
    assert label_col in df.columns, f'Missing label column: {label_col}'
    # Create a standardized view
    out = pd.DataFrame({
        'text': df[text_col].astype(str),
        'label_raw': df[label_col].astype(str)
    })
    if source_col in df.columns:
        out['source_type'] = df[source_col].astype(str)
    else:
        out['source_type'] = 'real'  # fallback
    return out

if SINGLE_CSV:
    base = pd.read_csv(DATA_PATH)
    base = standardize_columns(base, TEXT_COL, LABEL_COL, SOURCE_COL)
    # Encode labels (string -> int), save label map
    le = LabelEncoder()
    base['label'] = le.fit_transform(base['label_raw'])
    label_map = {int(i): cls for i, cls in enumerate(le.classes_)}
    os.makedirs('artifacts', exist_ok=True)
    with open('artifacts/label_map.json','w') as f:
        json.dump(label_map, f, indent=2)

    if REAL_ONLY_EVAL:
        real = base[base['source_type'] == 'real']
        rest = base
        real_train, real_holdout = train_test_split(real, test_size=0.20, random_state=SEED, stratify=real['label'])
        val_df, test_df = train_test_split(real_holdout, test_size=0.50, random_state=SEED, stratify=real_holdout['label'])
        used_idx = set(pd.concat([val_df, test_df]).index)
        train_df = rest[~rest.index.isin(used_idx)].copy()
    else:
        train_df, holdout = train_test_split(base, test_size=0.20, random_state=SEED, stratify=base['label'])
        val_df, test_df = train_test_split(holdout, test_size=0.50, random_state=SEED, stratify=holdout['label'])
else:
    train_df = standardize_columns(pd.read_csv(TRAIN_PATH), TEXT_COL, LABEL_COL, SOURCE_COL)
    val_df   = standardize_columns(pd.read_csv(VAL_PATH),   TEXT_COL, LABEL_COL, SOURCE_COL)
    test_df  = standardize_columns(pd.read_csv(TEST_PATH),  TEXT_COL, LABEL_COL, SOURCE_COL)
    # Fit label encoder on train and apply to all splits
    le = LabelEncoder()
    train_df['label'] = le.fit_transform(train_df['label_raw'])
    val_df['label']   = le.transform(val_df['label_raw'])
    test_df['label']  = le.transform(test_df['label_raw'])
    label_map = {int(i): cls for i, cls in enumerate(le.classes_)}
    os.makedirs('artifacts', exist_ok=True)
    with open('artifacts/label_map.json','w') as f:
        json.dump(label_map, f, indent=2)

num_labels = len(set(train_df['label']))
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | num_labels={num_labels}')
train_df.head()


Train: 51706 | Val: 2901 | Test: 2902 | num_labels=1


/var/folders/tf/_g_w7ph97cb_jcz6s2cpxt6c0000gn/T/ipykernel_27522/193829588.py:38: DtypeWarning: Columns (5,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = standardize_columns(pd.read_csv(TRAIN_PATH), TEXT_COL, LABEL_COL, SOURCE_COL)


,text,label_raw,source_type,label
0,sample text content for image_classification i...,Other,real,0
1,sample text content for image_classification i...,Other,real,0
2,sample text content for image_classification i...,Other,real,0
3,sample text content for image_classification i...,Other,real,0
4,sample text content for image_classification i...,Other,real,0


In [ ]:
## Tokenizer & Dataset

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['text']), truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.tensor(int(row['label']), dtype=torch.long)
        return item

train_ds = TextDataset(train_df, tokenizer, MAX_LEN)
val_ds   = TextDataset(val_df, tokenizer, MAX_LEN)
test_ds  = TextDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)
len(train_ds), len(val_ds), len(test_ds)


(51706, 2901, 2902)

## Model (BERT encoder + classifier)

In [12]:
class BertClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name, output_hidden_states=False)
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = outputs.last_hidden_state[:,0]
        logits = self.classifier(self.dropout(cls))
        if labels is not None:
            return logits, labels
        return logits

model = BertClassifier(MODEL_NAME, num_labels).to(DEVICE)
model


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

## Optimizer, Scheduler, Class Weights

In [13]:
cnt = Counter(train_df['label'])
total = sum(cnt.values())
class_weights = torch.tensor([total/(num_labels*cnt[i]) for i in range(num_labels)], dtype=torch.float32).to(DEVICE)
print('Class weights:', class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)


Class weights: tensor([1.])


## Train & Validate (Early Stopping on macro-F1)

In [ ]:
def run_epoch(dataloader, training=True):
    model.train() if training else model.eval()
    losses, all_preds, all_labels = [], [], []
    for batch in dataloader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None: token_type_ids = token_type_ids.to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        with torch.set_grad_enabled(training):
            logits, _ = model(input_ids, attention_mask, token_type_ids, labels)
            loss = loss_fn(logits, labels)
            if training:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
        losses.append(loss.item())
        preds = logits.argmax(dim=-1).detach().cpu().numpy()
        lbls = labels.detach().cpu().numpy()
        all_preds.extend(list(preds)); all_labels.extend(list(lbls))
    acc = accuracy_score(all_labels, all_preds)
    f1m = f1_score(all_labels, all_preds, average='macro')
    return np.mean(losses), acc, f1m

best_f1, patience = -1.0, 0
history = []
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, training=True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, training=False)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss, 'train_acc': tr_acc, 'val_acc': va_acc, 'train_f1': tr_f1, 'val_f1': va_f1})
    print(f"Epoch {epoch}: train_loss={tr_loss:.4f} val_loss={va_loss:.4f} | train_f1={tr_f1:.4f} val_f1={va_f1:.4f}")
    if va_f1 > best_f1:
        best_f1 = va_f1; patience = 0
        os.makedirs('artifacts', exist_ok=True)
        torch.save(model.state_dict(), 'artifacts/model.pt')
        with open('artifacts/history.json','w') as f: json.dump(history, f, indent=2)
    else:
        patience += 1
        if patience >= PATIENCE:
            print('Early stopping triggered.'); break
print('Best val F1:', best_f1)


## Test Evaluation

In [ ]:
model.load_state_dict(torch.load('artifacts/model.pt', map_location=DEVICE))
model.to(DEVICE); model.eval()
all_preds, all_labels = [], []
for batch in test_loader:
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    token_type_ids = batch.get('token_type_ids')
    if token_type_ids is not None: token_type_ids = token_type_ids.to(DEVICE)
    labels = batch['labels'].to(DEVICE)
    with torch.no_grad():
        logits, _ = model(input_ids, attention_mask, token_type_ids, labels)
        preds = logits.argmax(dim=-1).detach().cpu().numpy()
        lbls = labels.detach().cpu().numpy()
        all_preds.extend(list(preds)); all_labels.extend(list(lbls))
acc = accuracy_score(all_labels, all_preds)
f1m = f1_score(all_labels, all_preds, average='macro')
print({'test_accuracy': acc, 'test_f1_macro': f1m})
print('\nPer-class report:\n')
print(classification_report(all_labels, all_preds, digits=4))


## Inference Helper

In [ ]:
import torch.nn.functional as F

def load_label_map(path='artifacts/label_map.json'):
    with open(path) as f:
        return {int(k): v for k,v in json.load(f).items()}

def predict(texts, batch_size=32, return_labels=True):
    model.eval(); label_map = load_label_map()
    preds_all, probs_all = [], []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer(chunk, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k,v in enc.items()}
        with torch.no_grad():
            logits = model(**enc)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)
        if return_labels:
            preds_lbl = [label_map[int(p)] for p in preds]
            preds_all.extend(preds_lbl)
        else:
            preds_all.extend(preds.tolist())
        probs_all.extend(probs.tolist())
    return preds_all, probs_all

# Example:
# preds, probs = predict(["This is a sample.", "Another line of text."])
# preds, probs[:1]
